# 01: Generative Deep Learning: Variational Autoencoders (VAEs)

**Track 12: Generative Deep Learning (VAEs & Multimodal Search)** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Generative modeling in PyTorch: Encoder Gaussian parameterization (Mean $\mu$ and Log-Variance $\log\sigma^2$), Reparameterization Trick, KL Divergence loss, and latent image sampling.


## 1. Reparameterization Trick & Loss Formulation
$$\mathcal{L}_{\text{VAE}} = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{\text{KL}}(q_\phi(z|x) \parallel p(z))$$
where $z = \mu + \sigma \odot \epsilon$, with $\epsilon \sim \mathcal{N}(0, I)$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, input_dim=784, latent_dim=16):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        
        self.fc3 = nn.Linear(latent_dim, 256)
        self.fc4 = nn.Linear(256, input_dim)
        
    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def decode(self, z):
        h = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h))
        
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    bce = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld

vae = VAE(input_dim=784, latent_dim=16)
dummy_batch = torch.rand(8, 784)
recon, mu, logvar = vae(dummy_batch)
loss = vae_loss(recon, dummy_batch, mu, logvar)

print("=== VAE Architecture & Loss ===")
print(f"Input Shape : {dummy_batch.shape}")
print(f"Recon Shape : {recon.shape} | Latent Mu Shape: {mu.shape}")
print(f"Total ELBO Loss: {loss.item():.2f}")